In [1]:
# ============================================================
# APEXPLANET - TASK 2
# SQL & DATA EXTRACTION
# Complete SQL + Python Integration Project
# ============================================================

import sqlite3
import pandas as pd
import numpy as np
import os
import re
from pathlib import Path

print("=" * 70)
print("APEXPLANET - TASK 2: SQL & DATA EXTRACTION")
print("=" * 70)

# ============================================================
# 1. PROJECT PATHS
# ============================================================

NOTEBOOK_DIR = Path.cwd()

# When notebook is inside notebooks/, this points to project root
PROJECT_DIR = NOTEBOOK_DIR.parent

# If VS Code is running from project root instead, use current folder
if not (PROJECT_DIR / "data").exists():
    PROJECT_DIR = NOTEBOOK_DIR

DATA_FILE = PROJECT_DIR / "data" / "cleaned" / "sales.csv"

# Fallback locations
if not DATA_FILE.exists():
    possible_files = [
        PROJECT_DIR / "data" / "raw" / "sales.csv",
        PROJECT_DIR / "notebooks" / "cleaned_dataset.csv",
        NOTEBOOK_DIR / "cleaned_dataset.csv"
    ]

    for file in possible_files:
        if file.exists():
            DATA_FILE = file
            break

print("\nProject folder:", PROJECT_DIR)
print("Dataset:", DATA_FILE)

if not DATA_FILE.exists():
    raise FileNotFoundError(
        "sales.csv nahi mila. Please check data/cleaned/sales.csv"
    )

# ============================================================
# 2. IMPORT DATA
# ============================================================

df = pd.read_csv(DATA_FILE)

print("\n" + "=" * 70)
print("DATASET LOADED")
print("=" * 70)

print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

print("\nOriginal Columns:")
print(list(df.columns))

print("\nFirst 5 Rows:")
display(df.head())

# ============================================================
# 3. STANDARDIZE COLUMN NAMES
# ============================================================

def clean_column_name(column):
    column = str(column).strip().lower()
    column = re.sub(r"[^a-z0-9]+", "_", column)
    column = re.sub(r"_+", "_", column)
    return column.strip("_")

df.columns = [clean_column_name(col) for col in df.columns]

print("\nStandardized Columns:")
print(list(df.columns))

# ============================================================
# 4. REMOVE UNNECESSARY INDEX COLUMNS
# ============================================================

unwanted_columns = []

for col in df.columns:
    if col.startswith("unnamed") or col in ["index", "row_number"]:
        unwanted_columns.append(col)

if unwanted_columns:
    df.drop(columns=unwanted_columns, inplace=True)

print("\nColumns after cleaning:")
print(list(df.columns))

# ============================================================
# 5. AUTOMATIC COLUMN DETECTION
# ============================================================

def find_column(possible_names):
    for name in possible_names:
        if name in df.columns:
            return name
    return None

date_col = find_column([
    "date",
    "order_date",
    "sales_date",
    "transaction_date"
])

invoice_col = find_column([
    "invoice_id",
    "invoice",
    "invoice_no",
    "invoice_number",
    "transaction_id"
])

customer_col = find_column([
    "customer_id",
    "customer",
    "customer_name",
    "customer_id_number"
])

city_col = find_column([
    "city",
    "location"
])

customer_type_col = find_column([
    "customer_type",
    "customer_segment",
    "segment"
])

gender_col = find_column([
    "gender",
    "sex"
])

category_col = find_column([
    "category",
    "product_category"
])

product_col = find_column([
    "product",
    "product_name",
    "item",
    "item_name"
])

unit_price_col = find_column([
    "unit_price",
    "price",
    "unit_cost"
])

quantity_col = find_column([
    "quantity",
    "qty",
    "units"
])

sales_col = find_column([
    "total_sales",
    "sales",
    "revenue",
    "total_revenue",
    "amount",
    "total"
])

print("\n" + "=" * 70)
print("DETECTED COLUMNS")
print("=" * 70)

print("Date          :", date_col)
print("Invoice       :", invoice_col)
print("Customer      :", customer_col)
print("City          :", city_col)
print("Customer Type :", customer_type_col)
print("Gender        :", gender_col)
print("Category      :", category_col)
print("Product       :", product_col)
print("Unit Price    :", unit_price_col)
print("Quantity      :", quantity_col)
print("Sales         :", sales_col)

# ============================================================
# 6. DATA TYPE CONVERSION
# ============================================================

if date_col:
    df[date_col] = pd.to_datetime(
        df[date_col],
        errors="coerce"
    )

for col in [unit_price_col, quantity_col, sales_col]:
    if col:
        df[col] = pd.to_numeric(
            df[col],
            errors="coerce"
        )

# ============================================================
# 7. CREATE CUSTOMER KEY
# ============================================================

if customer_col:
    df["customer_key"] = df[customer_col].astype(str)
elif invoice_col:
    # If no customer ID exists, invoice becomes transaction/customer key
    df["customer_key"] = df[invoice_col].astype(str)
else:
    df["customer_key"] = df.index.astype(str)

# ============================================================
# 8. CREATE DATABASE
# ============================================================

DB_FILE = PROJECT_DIR / "task2_sales.db"

connection = sqlite3.connect(DB_FILE)

print("\n" + "=" * 70)
print("SQLITE DATABASE CREATED")
print("=" * 70)

print("Database:", DB_FILE)

# ============================================================
# 9. LOAD SALES DATA INTO SQLITE
# ============================================================

df.to_sql(
    "sales",
    connection,
    if_exists="replace",
    index=False
)

print("\nSales table created successfully.")

# ============================================================
# 10. CREATE CUSTOMER TABLE
# ============================================================

customer_columns = ["customer_key"]

for col in [customer_col, city_col, customer_type_col, gender_col]:
    if col and col not in customer_columns:
        customer_columns.append(col)

customers = df[customer_columns].drop_duplicates(
    subset=["customer_key"]
)

customers.to_sql(
    "customers",
    connection,
    if_exists="replace",
    index=False
)

print("Customers table created successfully.")

# ============================================================
# 11. CREATE PRODUCT TABLE
# ============================================================

product_columns = []

for col in [product_col, category_col, unit_price_col]:
    if col:
        product_columns.append(col)

if product_columns:
    products = df[product_columns].drop_duplicates()
    products.to_sql(
        "products",
        connection,
        if_exists="replace",
        index=False
    )

print("Products table created successfully.")

# ============================================================
# 12. HELPER FUNCTION FOR SQL
# ============================================================

def run_sql(query, title="SQL RESULT"):
    print("\n" + "=" * 70)
    print(title)
    print("=" * 70)

    try:
        result = pd.read_sql_query(
            query,
            connection
        )

        display(result)

        return result

    except Exception as error:
        print("SQL Error:", error)
        return None

# ============================================================
# 13. DATABASE TABLES
# ============================================================

run_sql(
    """
    SELECT name
    FROM sqlite_master
    WHERE type = 'table'
    ORDER BY name;
    """,
    "DATABASE TABLES"
)

# ============================================================
# 14. SELECT
# ============================================================

run_sql(
    """
    SELECT *
    FROM sales
    LIMIT 10;
    """,
    "1. SELECT - FIRST 10 RECORDS"
)

# ============================================================
# 15. WHERE
# ============================================================

run_sql(
    """
    SELECT *
    FROM sales
    WHERE quantity > 1
    LIMIT 10;
    """,
    "2. WHERE - QUANTITY GREATER THAN 1"
)

# ============================================================
# 16. ORDER BY
# ============================================================

if sales_col:

    run_sql(
        f"""
        SELECT *
        FROM sales
        ORDER BY {sales_col} DESC
        LIMIT 10;
        """,
        "3. ORDER BY - TOP SALES"
    )

# ============================================================
# 17. LIMIT
# ============================================================

run_sql(
    """
    SELECT *
    FROM sales
    LIMIT 5;
    """,
    "4. LIMIT - TOP 5 RECORDS"
)

# ============================================================
# 18. AGGREGATE FUNCTIONS
# ============================================================

if sales_col:

    run_sql(
        f"""
        SELECT
            COUNT(*) AS total_transactions,
            SUM({sales_col}) AS total_sales,
            AVG({sales_col}) AS average_sales,
            MIN({sales_col}) AS minimum_sales,
            MAX({sales_col}) AS maximum_sales
        FROM sales;
        """,
        "5. COUNT / SUM / AVG / MIN / MAX"
    )

# ============================================================
# 19. GROUP BY CATEGORY
# ============================================================

if category_col and sales_col:

    run_sql(
        f"""
        SELECT
            {category_col} AS category,
            COUNT(*) AS transactions,
            SUM({sales_col}) AS total_sales,
            AVG({sales_col}) AS average_sales
        FROM sales
        GROUP BY {category_col}
        ORDER BY total_sales DESC;
        """,
        "6. GROUP BY - CATEGORY PERFORMANCE"
    )

# ============================================================
# 20. HAVING
# ============================================================

if category_col and sales_col:

    run_sql(
        f"""
        SELECT
            {category_col} AS category,
            SUM({sales_col}) AS total_sales
        FROM sales
        GROUP BY {category_col}
        HAVING SUM({sales_col}) > 0
        ORDER BY total_sales DESC;
        """,
        "7. HAVING - CATEGORIES WITH SALES"
    )

# ============================================================
# 21. CITY-WISE ANALYSIS
# ============================================================

if city_col and sales_col:

    run_sql(
        f"""
        SELECT
            {city_col} AS city,
            COUNT(*) AS transactions,
            SUM({sales_col}) AS total_sales,
            AVG({sales_col}) AS average_sales
        FROM sales
        GROUP BY {city_col}
        ORDER BY total_sales DESC;
        """,
        "8. CITY-WISE SALES"
    )

# ============================================================
# 22. CUSTOMER TYPE ANALYSIS
# ============================================================

if customer_type_col and sales_col:

    run_sql(
        f"""
        SELECT
            {customer_type_col} AS customer_type,
            COUNT(*) AS transactions,
            SUM({sales_col}) AS total_sales,
            AVG({sales_col}) AS average_sales
        FROM sales
        GROUP BY {customer_type_col}
        ORDER BY total_sales DESC;
        """,
        "9. CUSTOMER TYPE ANALYSIS"
    )

# ============================================================
# 23. JOIN - SALES + CUSTOMERS
# ============================================================

run_sql(
    """
    SELECT
        s.customer_key,
        s.*
    FROM sales s
    INNER JOIN customers c
        ON s.customer_key = c.customer_key
    LIMIT 10;
    """,
    "10. INNER JOIN - SALES + CUSTOMERS"
)

# ============================================================
# 24. LEFT JOIN
# ============================================================

run_sql(
    """
    SELECT
        s.customer_key,
        c.customer_key AS matched_customer
    FROM sales s
    LEFT JOIN customers c
        ON s.customer_key = c.customer_key
    LIMIT 10;
    """,
    "11. LEFT JOIN"
)

# ============================================================
# 25. PRODUCT JOIN
# ============================================================

if product_col:

    run_sql(
        f"""
        SELECT
            s.{product_col} AS product,
            s.customer_key
        FROM sales s
        LEFT JOIN products p
            ON s.{product_col} = p.{product_col}
        LIMIT 10;
        """,
        "12. PRODUCT JOIN"
    )

# ============================================================
# 26. SUBQUERY
# ============================================================

if sales_col:

    run_sql(
        f"""
        SELECT *
        FROM sales
        WHERE {sales_col} >
        (
            SELECT AVG({sales_col})
            FROM sales
        )
        ORDER BY {sales_col} DESC;
        """,
        "13. SUBQUERY - ABOVE AVERAGE SALES"
    )

# ============================================================
# 27. CTE
# ============================================================

if category_col and sales_col:

    run_sql(
        f"""
        WITH category_sales AS
        (
            SELECT
                {category_col} AS category,
                SUM({sales_col}) AS total_sales
            FROM sales
            GROUP BY {category_col}
        )

        SELECT *
        FROM category_sales
        ORDER BY total_sales DESC;
        """,
        "14. CTE - CATEGORY SALES"
    )

# ============================================================
# 28. ROW_NUMBER
# ============================================================

if sales_col:

    run_sql(
        f"""
        SELECT
            customer_key,
            {sales_col} AS sales_amount,
            ROW_NUMBER() OVER (
                ORDER BY {sales_col} DESC
            ) AS row_number
        FROM sales;
        """,
        "15. ROW_NUMBER WINDOW FUNCTION"
    )

# ============================================================
# 29. RANK
# ============================================================

if sales_col:

    run_sql(
        f"""
        SELECT
            customer_key,
            {sales_col} AS sales_amount,
            RANK() OVER (
                ORDER BY {sales_col} DESC
            ) AS sales_rank
        FROM sales;
        """,
        "16. RANK WINDOW FUNCTION"
    )

# ============================================================
# 30. LAG
# ============================================================

if date_col and sales_col:

    run_sql(
        f"""
        SELECT
            {date_col} AS sale_date,
            {sales_col} AS sales_amount,
            LAG({sales_col}) OVER (
                ORDER BY {date_col}
            ) AS previous_sales
        FROM sales
        ORDER BY {date_col};
        """,
        "17. LAG WINDOW FUNCTION"
    )

# ============================================================
# 31. LEAD
# ============================================================

if date_col and sales_col:

    run_sql(
        f"""
        SELECT
            {date_col} AS sale_date,
            {sales_col} AS sales_amount,
            LEAD({sales_col}) OVER (
                ORDER BY {date_col}
            ) AS next_sales
        FROM sales
        ORDER BY {date_col};
        """,
        "18. LEAD WINDOW FUNCTION"
    )

# ============================================================
# 32. MONTHLY SALES
# ============================================================

if date_col and sales_col:

    run_sql(
        f"""
        SELECT
            strftime('%Y-%m', {date_col}) AS month,
            COUNT(*) AS transactions,
            SUM({sales_col}) AS monthly_sales
        FROM sales
        GROUP BY month
        ORDER BY month;
        """,
        "19. MONTHLY SALES ANALYSIS"
    )

# ============================================================
# 33. TOP 10 CUSTOMERS
# ============================================================

if sales_col:

    run_sql(
        f"""
        SELECT
            customer_key,
            COUNT(*) AS transactions,
            SUM({sales_col}) AS total_revenue,
            AVG({sales_col}) AS average_revenue
        FROM sales
        GROUP BY customer_key
        ORDER BY total_revenue DESC
        LIMIT 10;
        """,
        "20. TOP 10 CUSTOMERS BY REVENUE"
    )

# ============================================================
# 34. PRODUCT PERFORMANCE
# ============================================================

if product_col and sales_col:

    run_sql(
        f"""
        SELECT
            {product_col} AS product,
            COUNT(*) AS transactions,
            SUM({sales_col}) AS total_sales,
            AVG({sales_col}) AS average_sales
        FROM sales
        GROUP BY {product_col}
        ORDER BY total_sales DESC;
        """,
        "21. PRODUCT PERFORMANCE"
    )

# ============================================================
# 35. MOVING AVERAGE
# ============================================================

if date_col and sales_col:

    run_sql(
        f"""
        WITH daily_sales AS
        (
            SELECT
                {date_col} AS sale_date,
                SUM({sales_col}) AS daily_sales
            FROM sales
            GROUP BY {date_col}
        )

        SELECT
            sale_date,
            daily_sales,
            AVG(daily_sales) OVER (
                ORDER BY sale_date
                ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
            ) AS moving_average_3
        FROM daily_sales
        ORDER BY sale_date;
        """,
        "22. THREE-PERIOD MOVING AVERAGE"
    )

# ============================================================
# 36. CUMULATIVE SALES
# ============================================================

if date_col and sales_col:

    run_sql(
        f"""
        WITH daily_sales AS
        (
            SELECT
                {date_col} AS sale_date,
                SUM({sales_col}) AS daily_sales
            FROM sales
            GROUP BY {date_col}
        )

        SELECT
            sale_date,
            daily_sales,
            SUM(daily_sales) OVER (
                ORDER BY sale_date
            ) AS cumulative_sales
        FROM daily_sales
        ORDER BY sale_date;
        """,
        "23. CUMULATIVE SALES"
    )

# ============================================================
# 37. CUSTOMER RETENTION / REPEAT TRANSACTIONS
# ============================================================

run_sql(
    """
    SELECT
        customer_key,
        COUNT(*) AS number_of_transactions,
        CASE
            WHEN COUNT(*) > 1 THEN 'Returning Customer'
            ELSE 'One-Time Customer'
        END AS customer_status
    FROM sales
    GROUP BY customer_key
    ORDER BY number_of_transactions DESC;
    """,
    "24. CUSTOMER RETENTION / REPEAT TRANSACTIONS"
)

# ============================================================
# 38. CUSTOMER RETENTION SUMMARY
# ============================================================

run_sql(
    """
    WITH customer_activity AS
    (
        SELECT
            customer_key,
            COUNT(*) AS transactions
        FROM sales
        GROUP BY customer_key
    )

    SELECT
        customer_status,
        COUNT(*) AS customers
    FROM
    (
        SELECT
            customer_key,
            CASE
                WHEN transactions > 1
                THEN 'Returning Customer'
                ELSE 'One-Time Customer'
            END AS customer_status
        FROM customer_activity
    )
    GROUP BY customer_status;
    """,
    "25. CUSTOMER RETENTION SUMMARY"
)

# ============================================================
# 39. CREATE SQL VIEW
# ============================================================

if category_col and sales_col:

    connection.execute("DROP VIEW IF EXISTS category_sales_view")

    connection.execute(
        f"""
        CREATE VIEW category_sales_view AS
        SELECT
            {category_col} AS category,
            COUNT(*) AS transactions,
            SUM({sales_col}) AS total_sales,
            AVG({sales_col}) AS average_sales
        FROM sales
        GROUP BY {category_col};
        """
    )

    connection.commit()

    run_sql(
        """
        SELECT *
        FROM category_sales_view
        ORDER BY total_sales DESC;
        """,
        "26. SQL VIEW - CATEGORY SALES"
    )

# ============================================================
# 40. CREATE INDEX
# ============================================================

if customer_col or "customer_key" in df.columns:

    connection.execute(
        """
        CREATE INDEX IF NOT EXISTS idx_sales_customer
        ON sales(customer_key);
        """
    )

if category_col:

    connection.execute(
        f"""
        CREATE INDEX IF NOT EXISTS idx_sales_category
        ON sales({category_col});
        """
    )

if date_col:

    connection.execute(
        f"""
        CREATE INDEX IF NOT EXISTS idx_sales_date
        ON sales({date_col});
        """
    )

connection.commit()

print("\n" + "=" * 70)
print("INDEXES CREATED SUCCESSFULLY")
print("=" * 70)

# ============================================================
# 41. EXPLAIN QUERY PLAN
# ============================================================

if sales_col:

    run_sql(
        f"""
        EXPLAIN QUERY PLAN
        SELECT *
        FROM sales
        WHERE {sales_col} > 0;
        """,
        "27. EXPLAIN QUERY PLAN"
    )

# ============================================================
# 42. PARAMETERIZED QUERY
# ============================================================

if sales_col:

    parameter_query = f"""
    SELECT *
    FROM sales
    WHERE {sales_col} > ?
    ORDER BY {sales_col} DESC;
    """

    average_sales = pd.read_sql_query(
        f"""
        SELECT AVG({sales_col}) AS avg_sales
        FROM sales;
        """,
        connection
    ).iloc[0, 0]

    print("\nAverage sales used as parameter:", average_sales)

    parameter_result = pd.read_sql_query(
        parameter_query,
        connection,
        params=(average_sales,)
    )

    print("\n" + "=" * 70)
    print("28. PARAMETERIZED QUERY")
    print("=" * 70)

    display(parameter_result)

# ============================================================
# 43. PYTHON + SQL INTEGRATION
# ============================================================

if sales_col:

    python_sql_result = pd.read_sql(
        f"""
        SELECT
            COUNT(*) AS total_records,
            SUM({sales_col}) AS total_revenue,
            AVG({sales_col}) AS average_revenue,
            MIN({sales_col}) AS minimum_revenue,
            MAX({sales_col}) AS maximum_revenue
        FROM sales;
        """,
        connection
    )

    print("\n" + "=" * 70)
    print("29. PYTHON + SQL INTEGRATION")
    print("=" * 70)

    display(python_sql_result)

# ============================================================
# 44. SAVE SQL QUERY FILE
# ============================================================

SQL_FILE = NOTEBOOK_DIR / "task2_analysis.sql"

sql_script = r"""
-- ============================================================
-- APEXPLANET TASK 2
-- SQL & DATA EXTRACTION
-- ============================================================

-- 1. SELECT
SELECT *
FROM sales
LIMIT 10;

-- 2. WHERE
SELECT *
FROM sales
WHERE quantity > 1;

-- 3. ORDER BY
SELECT *
FROM sales
ORDER BY total_sales DESC;

-- 4. LIMIT
SELECT *
FROM sales
LIMIT 5;

-- 5. Aggregate Functions
SELECT
    COUNT(*) AS total_transactions,
    SUM(total_sales) AS total_sales,
    AVG(total_sales) AS average_sales,
    MIN(total_sales) AS minimum_sales,
    MAX(total_sales) AS maximum_sales
FROM sales;

-- 6. GROUP BY
SELECT
    category,
    COUNT(*) AS transactions,
    SUM(total_sales) AS total_sales
FROM sales
GROUP BY category
ORDER BY total_sales DESC;

-- 7. HAVING
SELECT
    category,
    SUM(total_sales) AS total_sales
FROM sales
GROUP BY category
HAVING SUM(total_sales) > 0;

-- 8. JOIN
SELECT
    s.customer_key,
    s.total_sales
FROM sales s
INNER JOIN customers c
    ON s.customer_key = c.customer_key;

-- 9. Subquery
SELECT *
FROM sales
WHERE total_sales >
(
    SELECT AVG(total_sales)
    FROM sales
);

-- 10. CTE
WITH category_sales AS
(
    SELECT
        category,
        SUM(total_sales) AS total_sales
    FROM sales
    GROUP BY category
)
SELECT *
FROM category_sales
ORDER BY total_sales DESC;

-- 11. ROW_NUMBER
SELECT
    customer_key,
    total_sales,
    ROW_NUMBER() OVER (
        ORDER BY total_sales DESC
    ) AS row_number
FROM sales;

-- 12. RANK
SELECT
    customer_key,
    total_sales,
    RANK() OVER (
        ORDER BY total_sales DESC
    ) AS sales_rank
FROM sales;

-- 13. LAG
SELECT
    date,
    total_sales,
    LAG(total_sales) OVER (
        ORDER BY date
    ) AS previous_sales
FROM sales;

-- 14. LEAD
SELECT
    date,
    total_sales,
    LEAD(total_sales) OVER (
        ORDER BY date
    ) AS next_sales
FROM sales;

-- 15. Monthly Sales
SELECT
    strftime('%Y-%m', date) AS month,
    SUM(total_sales) AS monthly_sales
FROM sales
GROUP BY month
ORDER BY month;

-- 16. Top 10 Customers
SELECT
    customer_key,
    SUM(total_sales) AS total_revenue
FROM sales
GROUP BY customer_key
ORDER BY total_revenue DESC
LIMIT 10;

-- 17. Product Performance
SELECT
    product,
    SUM(total_sales) AS total_sales
FROM sales
GROUP BY product
ORDER BY total_sales DESC;

-- 18. Moving Average
WITH daily_sales AS
(
    SELECT
        date,
        SUM(total_sales) AS daily_sales
    FROM sales
    GROUP BY date
)
SELECT
    date,
    daily_sales,
    AVG(daily_sales) OVER (
        ORDER BY date
        ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
    ) AS moving_average
FROM daily_sales;

-- 19. Cumulative Sales
WITH daily_sales AS
(
    SELECT
        date,
        SUM(total_sales) AS daily_sales
    FROM sales
    GROUP BY date
)
SELECT
    date,
    daily_sales,
    SUM(daily_sales) OVER (
        ORDER BY date
    ) AS cumulative_sales
FROM daily_sales;

-- 20. Customer Retention
SELECT
    customer_key,
    COUNT(*) AS transactions,
    CASE
        WHEN COUNT(*) > 1
        THEN 'Returning Customer'
        ELSE 'One-Time Customer'
    END AS customer_status
FROM sales
GROUP BY customer_key;

-- 21. View
CREATE VIEW IF NOT EXISTS category_sales_view AS
SELECT
    category,
    COUNT(*) AS transactions,
    SUM(total_sales) AS total_sales
FROM sales
GROUP BY category;

-- 22. Index
CREATE INDEX IF NOT EXISTS idx_sales_customer
ON sales(customer_key);

-- 23. EXPLAIN
EXPLAIN QUERY PLAN
SELECT *
FROM sales
WHERE total_sales > 100;
"""

with open(SQL_FILE, "w", encoding="utf-8") as file:
    file.write(sql_script)

print("\nSQL file created:")
print(SQL_FILE)

# ============================================================
# 45. CREATE db_utils.py
# ============================================================

DB_UTILS_FILE = NOTEBOOK_DIR / "db_utils.py"

db_utils_code = '''import sqlite3
from pathlib import Path
import pandas as pd


def get_connection(database_path="task2_sales.db"):
    """Create and return a SQLite database connection."""
    return sqlite3.connect(database_path)


def run_query(database_path, query, params=None):
    """Run a SQL query and return the result as a DataFrame."""
    connection = get_connection(database_path)

    try:
        return pd.read_sql_query(
            query,
            connection,
            params=params
        )
    finally:
        connection.close()


def execute_query(database_path, query, params=None):
    """Execute INSERT, UPDATE, DELETE or DDL queries."""
    connection = get_connection(database_path)

    try:
        connection.execute(query, params or ())
        connection.commit()
    finally:
        connection.close()
'''

with open(DB_UTILS_FILE, "w", encoding="utf-8") as file:
    file.write(db_utils_code)

print("\nPython database utility created:")
print(DB_UTILS_FILE)

# ============================================================
# 46. FINAL DATABASE CHECK
# ============================================================

print("\n" + "=" * 70)
print("FINAL DATABASE CHECK")
print("=" * 70)

tables = pd.read_sql_query(
    """
    SELECT
        name
    FROM sqlite_master
    WHERE type IN ('table', 'view')
    ORDER BY name;
    """,
    connection
)

display(tables)

# ============================================================
# 47. FINAL SUMMARY
# ============================================================

print("\n" + "=" * 70)
print("TASK 2 COMPLETED SUCCESSFULLY")
print("=" * 70)

print("""
Completed:
✓ SQLite database setup
✓ CSV imported into database
✓ SELECT
✓ WHERE
✓ ORDER BY
✓ LIMIT
✓ INNER JOIN
✓ LEFT JOIN
✓ GROUP BY
✓ HAVING
✓ COUNT
✓ SUM
✓ AVG
✓ MIN
✓ MAX
✓ Subqueries
✓ CTE
✓ ROW_NUMBER
✓ RANK
✓ LAG
✓ LEAD
✓ Monthly sales analysis
✓ Top 10 customers by revenue
✓ Customer retention analysis
✓ Product/category performance
✓ Moving average
✓ Cumulative sales
✓ SQL View
✓ Indexing
✓ EXPLAIN QUERY PLAN
✓ Parameterized query
✓ Python + SQL integration
✓ pandas.read_sql()
✓ db_utils.py
✓ SQL query file

Task 2 SQL & Data Extraction is ready.
""")

# Close database connection
connection.close()

print("Database connection closed.")
print("=" * 70)

APEXPLANET - TASK 2: SQL & DATA EXTRACTION

Project folder: c:\Users\HP\OneDrive\Desktop\Task_1_EDA
Dataset: c:\Users\HP\OneDrive\Desktop\Task_1_EDA\data\cleaned\sales.csv

DATASET LOADED
Rows: 21
Columns: 1

Original Columns:
['#\tA\tB\tC\tD\tE\tF\tG\tH\tI\tJ']

First 5 Rows:


,#\tA\tB\tC\tD\tE\tF\tG\tH\tI\tJ
0,1\tDate\tInvoice_ID\tCity\tCustomer_Type\tGend...
1,2\t2026-01-02\tA001\tMandalay\tMember\tFemale\...
2,3\t2026-01-03\tA002\tYangon\tNormal\tMale\tBev...
3,4\t2026-01-05\tA003\tNaypyitaw\tMember\tFemale...
4,5\t2026-01-07\tA004\tMandalay\tNormal\tFemale\...



Standardized Columns:
['a_b_c_d_e_f_g_h_i_j']

Columns after cleaning:
['a_b_c_d_e_f_g_h_i_j']

DETECTED COLUMNS
Date          : None
Invoice       : None
Customer      : None
City          : None
Customer Type : None
Gender        : None
Category      : None
Product       : None
Unit Price    : None
Quantity      : None
Sales         : None

SQLITE DATABASE CREATED
Database: c:\Users\HP\OneDrive\Desktop\Task_1_EDA\task2_sales.db

Sales table created successfully.
Customers table created successfully.
Products table created successfully.

DATABASE TABLES


,name
0,customers
1,sales



1. SELECT - FIRST 10 RECORDS


...


25. CUSTOMER RETENTION SUMMARY


,customer_status,customers
0,One-Time Customer,21



INDEXES CREATED SUCCESSFULLY

SQL file created:
c:\Users\HP\OneDrive\Desktop\Task_1_EDA\notebooks\task2_analysis.sql

Python database utility created:
c:\Users\HP\OneDrive\Desktop\Task_1_EDA\notebooks\db_utils.py

FINAL DATABASE CHECK


,name
0,customers
1,sales



TASK 2 COMPLETED SUCCESSFULLY

Completed:
✓ SQLite database setup
✓ CSV imported into database
✓ SELECT
✓ WHERE
✓ ORDER BY
✓ LIMIT
✓ INNER JOIN
✓ LEFT JOIN
✓ GROUP BY
✓ HAVING
✓ COUNT
✓ SUM
✓ AVG
✓ MIN
✓ MAX
✓ Subqueries
✓ CTE
✓ ROW_NUMBER
✓ RANK
✓ LAG
✓ LEAD
✓ Monthly sales analysis
✓ Top 10 customers by revenue
✓ Customer retention analysis
✓ Product/category performance
✓ Moving average
✓ Cumulative sales
✓ SQL View
✓ Indexing
✓ EXPLAIN QUERY PLAN
✓ Parameterized query
✓ Python + SQL integration
✓ pandas.read_sql()
✓ db_utils.py
✓ SQL query file

Task 2 SQL & Data Extraction is ready.

Database connection closed.
